# KIENDOAN-Studio · Nam Trầm Ấm

> **GPU:** T4 16GB · **Lần đầu:** ~5 phút · **Lần sau:** ~2 phút
> Chuyển phụ đề SRT thành giọng đọc AI — giữ nguyên timeline gốc

## [!] BẮT BUỘC: Bật GPU T4 thủ công

Colab **KHÔNG tự động cấp GPU** khi mở notebook từ GitHub. Bạn PHẢI tự bật:

1. **Runtime → Change runtime type → T4 GPU**
2. Sau đó mới **Runtime → Run all** (Ctrl+F9)

Nếu quên bước này, Cell 2 sẽ chờ 30s rồi in hướng dẫn — không crash, nhưng chạy trên CPU cực chậm (mỗi dòng TTS ~30-60s thay vì ~5s).

## Hướng dẫn

1. **Runtime → Change runtime type → T4 GPU** ← KHÔNG ĐƯỢC QUÊN
2. **Runtime → Run all** (Ctrl+F9) — đợi model load (~5 phút lần đầu)
3. Dán nội dung SRT (hoặc tải file .srt)
4. Tích chọn EBU R128 nếu muốn chuẩn hóa âm lượng
5. Click **[ Bắt đầu ]** → file MP3 tự động tải về

> [!] Giới hạn: ~500 dòng/lần (Colab T4 16GB RAM). Không upload nội dung nhạy cảm.
> Cần đổi giọng? Sửa biến `VOICE_KEY` ở Cell 1 — 7 giọng có sẵn trong repo voice-notebooks.

In [ ]:
# ============================================================
# CELL 1: Cai dat + Tai voice sample
# ============================================================

# ─── Cau hinh giong doc ─────
VOICE_KEY  = "nam-tram-am"
VOICE_NAME = "Nam tram am"
VOICE_URL  = "https://raw.githubusercontent.com/doanquangkien/voice-notebooks/main/samples/nam-tram-am.mp3"

print('Dang cai dat (~1 phut)...')
!pip install omnivoice gradio "numpy<2.1" "requests==2.32.4"
print('Cai dat hoan tat!')

# Tai giong mau 10s tu voice-notebooks repo
!wget -q {VOICE_URL} -O /tmp/voice_sample.mp3
print('Da tai voice sample!')

In [ ]:
# ============================================================
# CELL 2: Khoi dong Model + Pipeline + Giao dien
# ============================================================
# Cai ffmpeg (can cho trim + adelay + amix)
!apt-get install -qq ffmpeg
print("ffmpeg OK")

import logging, os, re, time, tempfile, subprocess, wave
import numpy as np
import torch
import IPython
from IPython.display import HTML, display, Javascript
from google.colab import output, files

# ─── Shims (bat buoc cho Colab) ──────────────────────
import torch as _torch
if not hasattr(_torch, '_utils'):
    _torch._utils = _torch._C._utils

import transformers as _tf
class _SafeAutoFeatureExtractor:
    @staticmethod
    def from_pretrained(model_name, **kwargs):
        try:
            from transformers import AutoConfig
            cfg = AutoConfig.from_pretrained(model_name, trust_remote_code=True, **kwargs)
            sr = getattr(cfg, 'sampling_rate', 24000)
        except Exception: sr = 24000
        class _Result: sampling_rate = sr
        return _Result()
_tf.AutoFeatureExtractor = _SafeAutoFeatureExtractor

from omnivoice import OmniVoice, OmniVoiceGenerationConfig
from omnivoice.utils.common import get_best_device

logging.basicConfig(level=logging.INFO, format='%(asctime)s %(levelname)s: %(message)s')
logger = logging.getLogger(__name__)

# ─── Cho GPU (30s poll — giong VOICE notebooks) ──────
print('Dang kiem tra GPU...')
gpu_found = False
for i in range(30):
    if torch.cuda.is_available():
        gpu_name = torch.cuda.get_device_name(0)
        gpu_found = True
        print(f'[OK] GPU: {gpu_name}')
        break
    time.sleep(1)
if not gpu_found:
    print('[!] GPU not available — chac ban chua bat T4.')
    print('-> Vao: Runtime > Change runtime type > T4 GPU')
    print('-> Sau do: Runtime > Run all (Ctrl+F9)')
    print('')
    print('Van tiep tuc chay tren CPU nhung se RAT CHAM...')
    print('(moi dong TTS ~30-60s thay vi ~5s tren GPU)')

# ─── Load model ──────────────────────────────────────
DEVICE = get_best_device()
logger.info(f'Loading OmniVoice on {DEVICE}...')
print(f'Dang tai OmniVoice tren {DEVICE}... (lan dau ~5 phut, cac lan sau ~30s)')
t_load = time.time()
model = OmniVoice.from_pretrained(
    'k2-fsa/OmniVoice', device_map=DEVICE, dtype=torch.float16, load_asr=True
)
SAMPLING_RATE = model.sampling_rate
print(f'[OK] Model san sang — Sample rate: {SAMPLING_RATE}Hz — {time.time()-t_load:.0f}s')

# ─── Tao Voice Clone Prompt (1 giọng duy nhat) ───────
print(f'\nDang tao voice clone prompt: {VOICE_NAME}...')
VOICE_PROMPT = model.create_voice_clone_prompt(ref_audio='/tmp/voice_sample.mp3')
print(f'[OK] Voice prompt san sang: {VOICE_NAME}')

# ─── Generation Config ────────────────────────────────
GEN_CFG = OmniVoiceGenerationConfig(
    num_step=32, guidance_scale=1.8,
    denoise=True, preprocess_prompt=True, postprocess_output=True,
    position_temperature=5.0, class_temperature=0.2,
    pad_duration=0.1, fade_duration=0.1,
)

# ─── Parse SRT ────────────────────────────────────────
def parse_srt(text):
    """Parse SRT text -> list[{index, start_s, end_s, text}]"""
    if text.startswith('﻿'):
        text = text[1:]
    # Chuan hoa line endings — file SRT Windows dung \r\n, regex can \n
    text = text.replace('\r\n', '\n').replace('\r', '\n')
    blocks = re.split(r'\n\s*\n', text.strip())
    lines = []
    def _ts(s):
        s = s.replace(',', '.')
        h, m, sec = s.split(':')
        return int(h)*3600 + int(m)*60 + float(sec)
    for block in blocks:
        m = re.match(
            r'(\d+)\s*\n\s*(\d{2}:\d{2}:\d{2}[,.]\d{3})\s*-->\s*'
            r'(\d{2}:\d{2}:\d{2}[,.]\d{3})[^\n]*\n\s*(.+)'  # [^\n]* = bo qua annotation tren dong timestamp
            block.strip(), re.DOTALL
        )
        if m:
            txt = re.sub(r'<[^>]+>', '', m.group(4).strip())
            lines.append({
                'index': int(m.group(1)),
                'start_s': _ts(m.group(2)),
                'end_s':   _ts(m.group(3)),
                'text':    txt.replace('\n', ' ')
            })
    return lines

# ─── WAV Writer (built-in wave module, khong can soundfile) ──
def write_wav(path, audio, sr):
    """Save numpy float audio [-1,1] as 16-bit mono WAV"""
    wf = (audio * 32767).astype(np.int16)
    with wave.open(path, 'w') as f:
        f.setnchannels(1); f.setsampwidth(2); f.setframerate(sr)
        f.writeframes(wf.tobytes())

# ─── JS Helpers ───────────────────────────────────────
def js_update(pct, text, eta_text=""):
    display(Javascript(f'''
        document.getElementById("progress-bar").style.width = "{pct}%";
        document.getElementById("progress-text").innerText = "{text}";
        document.getElementById("eta-text").innerText = "{eta_text}";
    '''), display_id=True)

def js_alert(msg):
    display(Javascript(f'alert("{msg}");'), display_id=True)
    print(msg)

# ─── Main Pipeline ────────────────────────────────────
def run_pipeline(srt_content, loudnorm=False):
    lines = parse_srt(srt_content)
    total = len(lines)

    if total == 0:
        return js_alert('Khong parse duoc dong SRT nao. Kiem tra lai dinh dang SRT.')
    if total > 500:
        return js_alert(f'{total} dong — qua nhieu! Gioi han 500 dong/lan de tranh het RAM.')

    tmpdir = tempfile.mkdtemp(prefix='srt_')
    timed_files = []
    t0 = time.time()

    js_update(0, f'{VOICE_NAME}: 0/{total} dong', 'Dang khoi dong TTS...')

    for i, line in enumerate(lines):
        tts_path   = os.path.join(tmpdir, f'tts_{i:04d}.wav')
        timed_path = os.path.join(tmpdir, f'timed_{i:04d}.wav')

        # 1. TTS — mỗi segment là 1 lần generate độc lập, không cần trim/warmup
        try:
            audio = model.generate(
                text=line['text'], voice_clone_prompt=VOICE_PROMPT,
                language='vi', speed=0.95, generation_config=GEN_CFG
            )[0]
            write_wav(tts_path, audio, SAMPLING_RATE)
        except Exception as e:
            print(f'  [!] TTS fail dong {i+1}: {e}')
            continue

        # 2. Resample + adelay — đặt audio vào đúng vị trí timeline
        #    KHÔNG trim audio — để giọng đọc tự nhiên, không cắt đầu/cuối
        delay_ms = int(line['start_s'] * 1000)

        r = subprocess.run([
            'ffmpeg', '-y', '-v', 'error',
            '-i', tts_path,
            '-af', (f'aresample={SAMPLING_RATE},'
                    f'aformat=sample_fmts=s16:channel_layouts=mono,'
                    f'adelay={delay_ms}:all=1'),
            '-c:a', 'pcm_s16le', timed_path
        ])
        if r.returncode != 0:
            os.remove(tts_path); continue

        timed_files.append(timed_path)
        os.remove(tts_path)

        # 3. Progress update
        if (i+1) % max(1, total//50) == 0 or i == total-1:
            pct = int((i+1)/total*100)
            e = time.time()-t0
            eta = (e/(i+1))*(total-i-1) if i>0 else 0
            js_update(pct, f'{i+1}/{total} dong ({pct}%) — {VOICE_NAME}',
                      f'Da chay: {int(e//60)}p{int(e%60)}s · Con lai: ~{int(eta//60)}p{int(eta%60)}s')

    if not timed_files:
        return js_alert('Pipeline that bai — khong co segment nao duoc tao. Kiem tra noi dung SRT.')

    # 4. Pairwise amix (2 inputs/lan — tranh OOM)
    js_update(100, f'Dang mix {len(timed_files)} segments...', '')
    current = timed_files[0]
    for j, nf in enumerate(timed_files[1:], 1):
        mixed = os.path.join(tmpdir, f'_mix_{j:04d}.wav')
        r = subprocess.run([
            'ffmpeg', '-y', '-v', 'error',
            '-i', current, '-i', nf,
            '-filter_complex', '[0:a][1:a]amix=inputs=2:duration=longest:normalize=0[a]',
            '-map', '[a]', '-c:a', 'pcm_s16le', mixed
        ])
        if r.returncode == 0:
            if os.path.exists(current): os.remove(current)
            if current != nf: os.remove(nf)
            current = mixed

    # 5. Convert WAV -> MP3
    output_mp3 = os.path.join(tmpdir, 'output.mp3')
    subprocess.run([
        'ffmpeg', '-y', '-v', 'error',
        '-i', current, '-codec:a', 'libmp3lame', '-b:a', '128k', output_mp3
    ])

    # 6. EBU R128 loudnorm (optional checkbox)
    if loudnorm:
        normalized = os.path.join(tmpdir, 'normalized.mp3')
        r = subprocess.run([
            'ffmpeg', '-y', '-v', 'error',
            '-i', output_mp3,
            '-af', 'loudnorm=I=-16:TP=-1.5:LRA=11',
            '-codec:a', 'libmp3lame', '-b:a', '128k', normalized
        ])
        if r.returncode == 0:
            output_mp3 = normalized

    # 7. Verify output
    probe = subprocess.run([
        'ffprobe', '-v', 'error', '-show_entries', 'format=duration',
        '-of', 'default=noprint_wrappers=1:nokey=1', output_mp3
    ], capture_output=True, text=True)
    adur = float(probe.stdout.strip()) if probe.returncode == 0 else 0
    smb  = os.path.getsize(output_mp3)/(1024*1024)
    tel  = time.time()-t0

    js_update(100, f'[OK] Hoan thanh! {adur:.0f}s audio · {smb:.1f}MB — Bam nut XANH ben duoi de tai',
              f'Tong thoi gian: {int(tel//60)}p{int(tel%60)}s · {total} dong · {VOICE_NAME}')

    import shutil
    final_path = "/tmp/SRT_Studio_output.mp3"
    shutil.copy2(output_mp3, final_path)
    shutil.rmtree(tmpdir, ignore_errors=True)
    print("[File] " + final_path + " (" + str(round(smb,1)) + "MB)")
    # Show download button in UI
    display(Javascript("document.getElementById('download-btn').style.display = 'block';"), display_id=True)
    norm_label = ' · EBU R128' if loudnorm else ''
    print(f'[OK] DONE — {total} dong · {smb:.1f}MB · {int(tel//60)}p{int(tel%60)}s{norm_label}')

# ─── Python Callback (goi tu JavaScript) ──────────────
def on_start(srt_text, loudnorm=False):
    display(Javascript('''
        document.getElementById("progress-section").style.display = "block";
        document.getElementById("start-btn").disabled = true;
        document.getElementById("start-btn").innerText = "Dang xu ly...";
    '''), display_id=True)
    run_pipeline(srt_text, loudnorm)
    display(Javascript('''
        document.getElementById("start-btn").disabled = false;
        document.getElementById("start-btn").innerText = "Bat dau tao giong doc";
    '''), display_id=True)

output.register_callback('on_start', on_start)

# Download callback (triggered by button click)
def do_download(_=None):
    files.download('/tmp/SRT_Studio_output.mp3')

output.register_callback('do_download', do_download)

# ─── HTML UI ──────────────────────────────────────────
display(HTML(f'''
<div style="font-family:Manrope,sans-serif;max-width:640px;margin:30px auto;
            background:#1a1a2e;border-radius:16px;padding:28px;color:#e0e0e0">
  <h2 style="color:#5B3DF6;margin:0 0 4px;font-size:20px">KIENDOAN-Studio</h2>
  <p style="margin:0 0 12px;color:#a0a0b0;font-size:14px">
     Chuyen phu de SRT thanh giong doc AI — giu nguyen timeline
  </p>
  <p style="margin:0 0 20px;padding:10px;border-radius:8px;
            background:#0f0f23;border:1px solid #2a2a4a;
            font-size:14px;color:#c0c0d0">
    <svg width='14' height='14' viewBox='0 0 24 24' fill='none' stroke='#5B3DF6' stroke-width='2' stroke-linecap='round' stroke-linejoin='round' style='vertical-align:middle;margin-right:5px'><path d='M12 2a3 3 0 0 0-3 3v7a3 3 0 0 0 6 0V5a3 3 0 0 0-3-3Z'/><path d='M19 10v2a7 7 0 0 1-14 0v-2'/><line x1='12' x2='12' y1='19' y2='22'/></svg>Giong doc: <strong style="color:#5B3DF6">{VOICE_NAME}</strong>
  </p>

  <label style="font-size:13px;color:#a0a0b0;margin-bottom:6px;display:block">
    Noi dung SRT:
  </label>
  <textarea id="srt-input"
    placeholder="1&#10;00:00:00,866 --&gt; 00:00:04,333&#10;Chao mung ban den voi video...&#10;&#10;2&#10;00:00:04,500 --&gt; 00:00:07,200&#10;Hom nay toi se gioi thieu..."
    style="width:100%;height:220px;border-radius:12px;padding:14px;
           font:13px 'JetBrains Mono',monospace;background:#0f0f23;
           color:#e0e0e0;border:1px solid #2a2a4a;resize:vertical;
           box-sizing:border-box"></textarea>

  <p style="text-align:center;color:#a0a0b0;margin:12px 0;font-size:13px">— hoac —</p>

  <button id="upload-btn"
    style="width:100%;padding:12px;border-radius:12px;border:1.5px dashed #5B3DF6;
           background:transparent;color:#5B3DF6;font:14px Manrope,sans-serif;
           cursor:pointer">
    [ Chon file .srt ]
  </button>

  <div style="margin-top:14px;display:flex;align-items:center;gap:10px;
              padding:10px 14px;background:#0f0f23;border-radius:10px;
              border:1px solid #2a2a4a">
    <input type="checkbox" id="loudnorm-cb"
           style="width:18px;height:18px;accent-color:#5B3DF6;cursor:pointer">
    <label for="loudnorm-cb"
           style="font-size:13px;color:#c0c0d0;cursor:pointer;user-select:none">
      Chuan hoa am luong EBU R128 (-16 LUFS)
    </label>
  </div>

  <button id="start-btn"
    style="width:100%;margin-top:14px;padding:14px;border-radius:12px;
           border:none;background:#5B3DF6;color:#fff;font:15px Manrope,sans-serif;
           font-weight:600;cursor:pointer">
    [ Bat dau ]
  </button>

  <button id="download-btn" onclick="google.colab.kernel.invokeFunction('do_download', [], {{}})" style="width:100%;margin-top:14px;padding:14px;border-radius:12px;border:none;background:#00C853;color:#fff;font:15px Manrope,sans-serif;font-weight:600;cursor:pointer;display:none">[ Tai xuong ]</button>

  <div id="progress-section" style="display:none;margin-top:20px">
    <div style="background:#2a2a4a;border-radius:8px;height:10px;overflow:hidden">
      <div id="progress-bar"
        style="background:linear-gradient(90deg,#5B3DF6,#8B6FFF);width:0%;
               height:100%;border-radius:8px;transition:width .3s"></div>
    </div>
    <p id="progress-text" style="text-align:center;margin-top:10px;font-size:14px;color:#c0c0d0"></p>
    <p id="eta-text" style="text-align:center;font-size:13px;color:#8080a0"></p>
  </div>

  <p style="text-align:center;margin-top:20px;font-size:12px;color:#606080">
    GPU T4 · OmniVoice · KIENDOAN-Studio
  </p>
</div>

<script>
document.getElementById('upload-btn').onclick = function() {{
    var input = document.createElement('input');
    input.type = 'file';
    input.accept = '.srt,.txt';
    input.onchange = function(e) {{
        var file = e.target.files[0];
        if (!file) return;
        var reader = new FileReader();
        reader.onload = function(e) {{
            document.getElementById('srt-input').value = e.target.result;
        }};
        reader.readAsText(file);
    }};
    input.click();
}};

document.getElementById('start-btn').onclick = async function() {{
    var srt = document.getElementById('srt-input').value;
    if (!srt.trim()) {{
        alert('[!] Vui long nhap noi dung SRT hoac tai file .srt len truoc.');
        return;
    }}
    var loudnorm = document.getElementById('loudnorm-cb').checked;
    await google.colab.kernel.invokeFunction('on_start', [srt, loudnorm], {{}});
}};
</script>
'''))

if torch.cuda.is_available():
    print(f'[OK] San sang! Giong: {VOICE_NAME} | GPU: {torch.cuda.get_device_name(0)}')
else:
    print(f'[!] San sang! Giong: {VOICE_NAME} | CPU (rat cham — nen bat T4 GPU)')
print('-> Dan SRT vao o textarea (hoac tai file) → Bam "Bat dau tao giong doc"')